# EMA pilot: conditions A vs C on Qwen3-4B

Runs the pilot slice from the README: 20 `conflicting` Impossible-LiveCodeBench tasks, memory conditions **A** (none) and **C** (failure), 1 epoch, Qwen3-4B-Instruct-2507.
This checks the pipeline and looks for floor effects. **It is not a hypothesis test.**

**Kaggle setup:** Accelerator *GPU T4 x2*, Internet *on*. Colab works with one T4: set `TP = 1` below.

The run uses Inspect's `local` sandbox, so model-written code runs on this VM. That is acceptable on a throwaway notebook VM, not on your own machine.

If the session dies, re-run all cells. `eval_set` resumes from the logs already in `LOG_DIR`.

In [ ]:
import os, shutil, subprocess, sys, torch
REPO = "https://github.com/NischalParulekar/EMA"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
TP = max(1, torch.cuda.device_count())   # tensor-parallel degree = number of GPUs
LIMIT, CONDITIONS, MAX_ATTEMPTS = 20, ["A", "C"], 5
LOG_DIR = f"{WORK}/logs/pilot_qwen3-4b"
PY = sys.executable       # Kaggle/Colab Python is 3.12+, which ImpossibleBench needs
os.chdir(WORK)
print(WORK, "GPUs:", TP)

## 1. Environment
Installs into the image's own Python (Kaggle: 3.13; ImpossibleBench needs 3.12+). This takes a few minutes.

In [ ]:
%%bash -s "$REPO" "$WORK" "$PY"
set -e
cd "$2"
[ -d EMA ] || git clone -q "$1" EMA
pip install -q uv
uv pip install -q --python "$3" vllm -e "./EMA[dev]"
uv pip install -q --python "$3" --no-deps -r EMA/requirements-impossiblebench.txt
git -C EMA rev-parse HEAD
"$3" --version
uv pip freeze --python "$3" > EMA/results/pilot_env.txt
grep -iE "^(vllm|inspect-ai|torch|transformers|impossiblebench)" EMA/results/pilot_env.txt

## 2. Serve the model with vLLM
T4s have no bf16, so `--dtype half`.

In [ ]:
VLLM = shutil.which("vllm") or os.path.join(os.path.dirname(PY), "vllm")
server = subprocess.Popen(
    [VLLM, "serve", MODEL, "--dtype", "half", "--tensor-parallel-size", str(TP),
     "--max-model-len", "16384", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open(f"{WORK}/vllm.log", "w"), stderr=subprocess.STDOUT,
)

In [ ]:
import time, urllib.request
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2); print("vLLM ready"); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(open(f"{WORK}/vllm.log").read()[-3000:])
        time.sleep(10)
else:
    raise TimeoutError("vLLM did not come up; see vllm.log")

If vLLM will not start on this GPU, skip the two cells above and set `INSPECT_MODEL = f"hf/{MODEL}"` in the run cell below (`run_pilot.py` then loads it in float16). Inspect's Hugging Face provider is slower but needs no server.

## 3. Run conditions A and C

In [ ]:
%%writefile run_pilot.py
import os, sys
from inspect_ai import eval_set
from ema.tasks.memory_lcb import memory_lcb

log_dir, model, limit, max_attempts, *conditions = sys.argv[1:]
os.environ.setdefault("VLLM_BASE_URL", "http://localhost:8000/v1")
os.environ.setdefault("VLLM_API_KEY", "local")
tasks = [memory_lcb(condition=c, limit=int(limit), max_attempts=int(max_attempts)) for c in conditions]
ok, logs = eval_set(
    tasks,
    model=model,
    model_args={"torch_dtype": "float16"} if model.startswith("hf/") else {},
    log_dir=log_dir,
    temperature=0.7,
    max_tokens=2048,
    max_connections=8,
    max_subprocesses=8,
    fail_on_error=False,
    display="plain",
)
print("all tasks complete:", ok)

In [ ]:
INSPECT_MODEL = f"openai-api/vllm/{MODEL}"   # served by the vLLM cell above
!PYTHONIOENCODING=utf-8 {PY} run_pilot.py {LOG_DIR} {INSPECT_MODEL} {LIMIT} {MAX_ATTEMPTS} {" ".join(CONDITIONS)}

## 4. Summarise and save to `results/`

In [ ]:
!{PY} -m ema.analysis {LOG_DIR} {WORK}/EMA/results/pilot_qwen3-4b.csv
!mkdir -p {WORK}/EMA/results/logs && cp {LOG_DIR}/*.eval {WORK}/EMA/results/logs/ && du -sh {WORK}/EMA/results

## 5. Read the transcripts before believing the number

Open a few samples with `inspect view --log-dir {LOG_DIR}`, or read the `.eval` files with `inspect_ai.log.read_eval_log`. Check that every sample the detector labels is actually a cheat, and that no unlabelled pass is a cheat in disguise.

Then download `EMA/results/` (pilot CSV, `.eval` logs, `pilot_env.txt`), commit it to the repo and push.

In [ ]:
server.terminate()